This notebook populates a neo4j Graph Database with Pokemon information fetched from the pokeapi API (up to date until Gen 9).

In [1]:
import requests
from neo4j import GraphDatabase
from pathlib import Path
import os
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
import time

In [2]:
dotenv_path = Path('~/.env').expanduser()
load_dotenv(dotenv_path=dotenv_path)  

# Get variables
POKEDEX_URI = os.getenv('POKEDEX_URI')
POKEDEX_USER = os.getenv('POKEDEX_USER')
POKEDEX_PASSWORD = os.getenv('POKEDEX_PASSWORD')
database_name = os.getenv('DATABASE_NAME')

driver = GraphDatabase.driver(POKEDEX_URI, auth=(POKEDEX_USER, POKEDEX_PASSWORD))
driver.verify_connectivity()

In [3]:
# define type effectiveness dictionaries

type_effectiveness = {
    "Normal" : [],
    "Fire" : ["Grass", "Ice", "Bug", "Steel"],
    "Water" : ["Fire", "Ground", "Rock"],
    "Electric" : ["Water", "Flying"],
    "Grass" : ["Water", "Ground", "Rock"],
    "Ice" : ["Grass", "Ground", "Flying", "Dragon"],
    "Fighting" : ["Normal", "Ice", "Rock", "Dark", "Steel"],
    "Poison" : ["Grass", "Fairy"],
    "Ground" : ["Fire", "Electric", "Poison", "Rock", "Steel"],
    "Flying" : ["Grass", "Fighting", "Bug"],
    "Psychic" : ["Fighting", "Poison"],
    "Bug" : ["Grass", "Psychic", "Dark"],
    "Rock" : ["Fire", "Ice", "Flying", "Bug"],
    "Ghost" : ["Psychic", "Ghost"],
    "Dragon" : ["Dragon"],
    "Dark" : ["Psychic", "Ghost"],
    "Steel" : ["Ice", "Rock", "Fairy"],
    "Fairy" : ["Fighting", "Dragon", "Dark"]
}

type_not_very_effectiveness = {
    "Normal" : ["Rock", "Steel"],
    "Fire" : ["Fire", "Water", "Rock", "Dragon"],
    "Water" : ["Water", "Grass", "Dragon"],
    "Electric" : ["Electric", "Grass", "Dragon"],
    "Grass" : ["Fire", "Grass", "Poison", "Flying", "Bug", "Dragon", "Steel"],
    "Ice" : ["Fire", "Water", "Ice", "Steel"],
    "Fighting" : ["Poison", "Flying", "Psychic", "Bug", "Fairy"],
    "Poison" : ["Poison", "Ground", "Rock", "Ghost"],
    "Ground" : ["Grass", "Bug"],
    "Flying" : ["Electric", "Rock", "Steel"],
    "Psychic" : ["Psychic", "Steel"],
    "Bug" : ["Fire", "Fighting", "Poison", "Flying", "Ghost", "Steel", "Fairy"],
    "Rock" : ["Fighting", "Ground", "Steel"],
    "Ghost" : ["Dark"],
    "Dragon" : ["Steel"],
    "Dark" : ["Fighting", "Dark", "Fairy"],
    "Steel" : ["Fire", "Water", "Electric", "Steel"],
    "Fairy" : ["Fire", "Poison", "Steel"] 
}

type_immunity = {
    "Normal" : ["Ghost"],
    "Fire" : [],
    "Water" : [],
    "Electric" : ["Ground"],
    "Grass" : [],
    "Ice" : [],
    "Fighting" : ["Ghost"],
    "Poison" : ["Steel"],
    "Ground" : ["Flying"],
    "Flying" : [],
    "Psychic" : ["Dark"],
    "Bug" : [],
    "Rock" : [],
    "Ghost" : ["Normal"],
    "Dragon" : ["Fairy"],
    "Dark" : [],
    "Steel" : [],
    "Fairy" : []
}

> Fetch data

In [ ]:
def fetch_pokemon_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

base_url = "https://pokeapi.co/api/v2/pokemon"
data = fetch_pokemon_data(base_url)
pokemon_list = []

# iterate through the pages and get data for each pokemon
while data:
    for pokemon in data['results']:
        pokemon_data = fetch_pokemon_data(pokemon['url'])
        time.sleep(0.2)

        species_data = fetch_pokemon_data(pokemon_data["species"]['url'])

        name = pokemon_data["name"].capitalize()
        types = [t["type"]["name"].capitalize() for t in pokemon_data["types"]]
        generation = species_data["generation"]["name"].capitalize()

        # pokedex number
        pokedex_number = None
        for entry in species_data["pokedex_numbers"]:   
            if entry["pokedex"]["name"] == "national":
                pokedex_number = entry["entry_number"]
                break

        description = None
        for entry in species_data["flavor_text_entries"]:
            if entry["language"]["name"] == "en":
                description = entry["flavor_text"].replace("\n", " ").replace("\f", " ")
                break

        # abilities
        abilities = [a["ability"]["name"].capitalize() for a in pokemon_data["abilities"]]

        # moves
        moves = [m["move"]["name"].capitalize() for m in pokemon_data["moves"]]

        pokemon_list.append({
            "name": name,
            "pokedex_number": pokedex_number,
            "description": description,
            "types": types,
            "generation": generation,
            "abilities": abilities,
            "moves": moves
        })

    # get the next page (if any)
    next_url = data['next']
    if next_url:
        data = fetch_pokemon_data(next_url)
    else:
        break

# create collective df
df = pd.DataFrame(pokemon_list)
df['name'] = df['name'].str.capitalize()
df.head()


In [ ]:
#evolutionary line
df['evolves_from'] = None

# loop through Pokemon and fetch evolution info
for idx, row in tqdm(df.iterrows(), total=len(df)):
    name = row['name'].lower()
    url = f"https://pokeapi.co/api/v2/pokemon-species/{name}"
    response = requests.get(url)

    if response.status_code == 200:
        species_data = response.json()
        evolves_from = species_data.get('evolves_from_species')
        if evolves_from:
            df.at[idx, 'evolves_from'] = evolves_from['name'].capitalize()

df.head()

100%|██████████| 1302/1302 [04:47<00:00,  4.54it/s]


,name,types,generation,abilities,moves,evolves_from
0,Bulbasaur,"[grass, poison]",generation-i,"[overgrow, chlorophyll]","[razor-wind, swords-dance, cut, bind, vine-whi...",None
1,Ivysaur,"[grass, poison]",generation-i,"[overgrow, chlorophyll]","[swords-dance, cut, bind, vine-whip, headbutt,...",Bulbasaur
2,Venusaur,"[grass, poison]",generation-i,"[overgrow, chlorophyll]","[swords-dance, cut, bind, vine-whip, headbutt,...",Ivysaur
3,Charmander,[fire],generation-i,"[blaze, solar-power]","[mega-punch, fire-punch, thunder-punch, scratc...",None
4,Charmeleon,[fire],generation-i,"[blaze, solar-power]","[mega-punch, fire-punch, thunder-punch, scratc...",Charmander


In [ ]:
# ability dataframe
ability_base_url = "https://pokeapi.co/api/v2/ability/"

def fetch_abilities():
    abilities = []
    url = ability_base_url
    while url:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        abilities.extend(data["results"])
        url = data["next"]
        time.sleep(0.1)  # avoid hammering the API
    return abilities

# for ability description
def fetch_ability_details(ability_url):
    response = requests.get(ability_url)
    response.raise_for_status()
    data = response.json()

    effect = None
    short_effect = None
    for entry in data["effect_entries"]:
        if entry["language"]["name"] == "en":
            effect = entry.get("effect")
            short_effect = entry.get("short_effect")
            break

    return {
        "name": data["name"],
        "effect": effect,
        "short_effect": short_effect
    }

all_abilities = fetch_abilities()

ability_data = []
for ability in all_abilities:
    details = fetch_ability_details(ability["url"])
    ability_data.append(details)

df_abilities = pd.DataFrame(ability_data)
df_abilities.head()

,name,effect,short_effect
0,stench,This Pokémon's damaging moves have a 10% chanc...,Has a 10% chance of making target Pokémon flin...
1,drizzle,The weather changes to rain when this Pokémon ...,Summons rain that lasts indefinitely upon ente...
2,speed-boost,This Pokémon's Speed rises one stage after eac...,Raises Speed one stage after each turn.
3,battle-armor,Moves cannot score critical hits against this ...,Protects against critical hits.
4,sturdy,"When this Pokémon is at full HP, any hit that ...","Prevents being KOed from full HP, leaving 1 HP..."


In [ ]:
# move dataframe
move_base_url = "https://pokeapi.co/api/v2/move/"

def fetch_moves():
    moves = []
    url = move_base_url
    while url:
        response = requests.get(url)
        data = response.json()
        moves.extend(data["results"])
        url = data["next"]  # Next page
    return moves

def fetch_move_details(move_url):
    response = requests.get(move_url)
    data = response.json()

    # Extract description (English flavor text)
    description = None
    for entry in data["flavor_text_entries"]:
        if entry["language"]["name"] == "en":
            description = entry["flavor_text"].replace("\n", " ").replace("\f", " ")
            break

    return {
        "name": data["name"],
        "type": data["type"]["name"],
        "accuracy": data["accuracy"],
        "power": data["power"],
        "pp": data["pp"],
        "description": description
    }

all_moves = fetch_moves()

move_data = []
for move in all_moves:
    details = fetch_move_details(move["url"])
    move_data.append(details)

df_moves = pd.DataFrame(move_data)
df_moves["name"] = df_moves["name"].str.capitalize()
df_moves["type"] = df_moves["type"].str.capitalize()
df_moves.head()

,name,type,accuracy,power,pp,description
0,Pound,Normal,100.0,40.0,35.0,Pounds with fore­ legs or tail.
1,Karate-chop,Fighting,100.0,50.0,25.0,Has a high criti­ cal hit ratio.
2,Double-slap,Normal,85.0,15.0,10.0,Repeatedly slaps 2-5 times.
3,Comet-punch,Normal,85.0,18.0,15.0,Repeatedly punches 2-5 times.
4,Mega-punch,Normal,85.0,80.0,20.0,A powerful punch thrown very hard.


> KG population

In [ ]:
def write_data(tx,statement, params_dict):
    tx.run(statement, parameters=params_dict)

In [ ]:
statement_create_pokemon = """MERGE (p:Pokemon {name:$name})"""

with driver.session() as session:
    for index, row in df.iterrows():
        session.execute_write(write_data, 
                                params_dict = {
                                    'name':str(row['name'])
                                },
                                statement = statement_create_pokemon)  

In [ ]:
statement_set_pokemon_info = """MATCH (p:Pokemon {name:$name}) SET p.number=$number, p.description=$description"""

with driver.session() as session:
    for index, row in df.iterrows():
        session.execute_write(write_data, 
                                params_dict = {
                                    'name':str(row['name']),
                                    'number':row['pokedex_number'],
                                    'description':row['description']
                                },
                                statement = statement_set_pokemon_info)  

In [ ]:
statement_create_types = """MERGE (t:Type {name:$type})"""

with driver.session() as session:
    for index, row in df.iterrows():
        for type_name in row['types']: 
            session.execute_write(
                write_data, 
                params_dict={
                    'type': str(type_name).capitalize()  # capitalize type names
                },
                statement=statement_create_types
            )

In [ ]:
statement_create_generation = """MERGE (g:Generation {number:$generation})"""

with driver.session() as session:
    for generation_number in df['generation']:  
        session.execute_write(
            write_data, 
            params_dict={
                'generation': str(generation_number) 
            },
                statement=statement_create_generation
        )

In [ ]:
statement_create_moves = """MERGE (m:Move {name: $name})
SET m.type=$type, m.accuracy=$accuracy, m.power=$power, m.pp=$pp, m.description=$description
"""
with driver.session() as session:
    for name, typing, accuracy, power, pp, description in zip(df_moves['name'], df_moves['type'], df_moves['accuracy'], df_moves['power'], df_moves['pp'], df_moves['description']):
        session.execute_write(
                write_data, 
                params_dict={
                    'name': name,
                    'type' : typing,
                    'accuracy' : accuracy,
                    'power' : power,
                    'pp' : pp,
                    'description' : description
                },
                statement=statement_create_moves
        )

In [ ]:
statement_has_type = """MATCH (p:Pokemon {name: $name})
MATCH (t:Type {name:$type})
MERGE (p)-[:HAS_TYPE]->(t)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        for type_name in row['types']:
            session.execute_write(
                write_data,
                params_dict = {
                    'name': pokemon_name,
                    'type': type_name
                },
                statement = statement_has_type
            )

C:\Users\Eleftheria\AppData\Local\Temp\ipykernel_19308\3795341116.py:6: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


In [ ]:
statement_has_type_move = """MATCH (m:Move {name: $name})
MATCH (t:Type {name:$type})
MERGE (m)-[:HAS_TYPE]->(t)
"""
with driver.session() as session:
    for name, type in zip(df_moves['name'], df_moves['type']):
        session.execute_write(
                write_data,
                params_dict = {
                    'name': name,
                    'type': type
                },
                statement = statement_has_type_move
        )

In [ ]:
statement_create_abilities = """
MERGE (a:Ability {name:$name})
SET a.effect = $effect, a.short_effect = $short_effect
"""

with driver.session() as session:
    for name, effect, short_effect in zip(df_abilities['name'], df_abilities['effect'], df_abilities['short_effect']):
        session.execute_write(
                write_data, 
                params_dict={
                    'name': str(name).capitalize(),
                    'effect': str(effect),
                    'short_effect' : str(short_effect)
                },
                statement=statement_create_abilities
        )

In [ ]:
statement_has_ability = """MATCH (p:Pokemon {name: $name})
MATCH (a:Ability {name:$ability})
MERGE (p)-[:HAS_ABILITY]->(a)
"""

with driver.session() as session:
    for pokemon, abilities in zip(df['name'], df['abilities']):
        for ability in abilities:
            session.execute_write(
                write_data,
                params_dict = {
                    'name': pokemon,
                    'ability': ability
                },
                statement = statement_has_ability
            )

In [ ]:
statement_in_generation = """MATCH (p:Pokemon {name: $name})
MATCH (g:Generation {number:$generation})
MERGE (p)-[:INTRODUCED_IN]->(g)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        generation_number = row['generation']
        session.execute_write(
                write_data,
                params_dict = {
                    'name': pokemon_name,
                    'generation': generation_number
                },
                statement = statement_in_generation
        )

In [ ]:
statement_evolves_from = """MATCH (p1:Pokemon {name: $name1})
MATCH (p2:Pokemon {name: $name2})
MERGE (p2)<-[:EVOLVES_FROM]-(p1)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        evolves_from = row['evolves_from']
        if (evolves_from != 'None'):
            session.execute_write(
                    write_data,
                    params_dict = {
                        'name1': pokemon_name,
                        'name2': evolves_from
                    },
                    statement = statement_evolves_from
            )

In [ ]:
statement_effectiveness = """MATCH (t1: Type {name: $type1})
MATCH (t2:Type {name:$type2})
MERGE (t1)-[:EFFECTIVENESS {factor:$factor}]->(t2)
"""

with driver.session() as session:
    for key, values in type_effectiveness.items():
        for value in values: 
            session.execute_write(
                write_data,
                params_dict = {
                    'type1': key,
                    'type2': value,
                    'factor' : 2
                },
                statement = statement_effectiveness
            )
    for key, values in type_not_very_effectiveness.items():
        for value in values: 
            session.execute_write(
                write_data,
                params_dict = {
                    'type1': key,
                    'type2': value,
                    'factor' : 0.5
                },
                statement = statement_effectiveness
            )
    for key, values in type_immunity.items():
        for value in values: 
            session.execute_write(
                write_data,
                params_dict = {
                    'type1': key,
                    'type2': value,
                    'factor' : 0
                },
                statement = statement_effectiveness
            )

In [ ]:
statement_has_move = """MATCH (p:Pokemon {name: $name})
MATCH (m:Move {name:$move})
MERGE (p)-[:HAS_MOVE]->(m)
"""
with driver.session() as session:
    for pokemon, moves in zip(df['name'], df['moves']):
        print(pokemon)
        for move in moves:
            session.execute_write(
                write_data,
                params_dict = {
                    'name': pokemon,
                    'move': move
                },
                statement = statement_has_move
            )

Bulbasaur
Ivysaur
Venusaur
Charmander
Charmeleon
Charizard
Squirtle
Wartortle
Blastoise
Caterpie
Metapod
Butterfree
Weedle
Kakuna
Beedrill
Pidgey
Pidgeotto
Pidgeot
Rattata
Raticate
Spearow
Fearow
Ekans
Arbok
Pikachu
Raichu
Sandshrew
Sandslash
Nidoran-f
Nidorina
Nidoqueen
Nidoran-m
Nidorino
Nidoking
Clefairy
Clefable
Vulpix
Ninetales
Jigglypuff
Wigglytuff
Zubat
Golbat
Oddish
Gloom
Vileplume
Paras
Parasect
Venonat
Venomoth
Diglett
Dugtrio
Meowth
Persian
Psyduck
Golduck
Mankey
Primeape
Growlithe
Arcanine
Poliwag
Poliwhirl
Poliwrath
Abra
Kadabra
Alakazam
Machop
Machoke
Machamp
Bellsprout
Weepinbell
Victreebel
Tentacool
Tentacruel
Geodude
Graveler
Golem
Ponyta
Rapidash
Slowpoke
Slowbro
Magnemite
Magneton
Farfetchd
Doduo
Dodrio
Seel
Dewgong
Grimer
Muk
Shellder
Cloyster
Gastly
Haunter
Gengar
Onix
Drowzee
Hypno
Krabby
Kingler
Voltorb
Electrode
Exeggcute
Exeggutor
Cubone
Marowak
Hitmonlee
Hitmonchan
Lickitung
Koffing
Weezing
Rhyhorn
Rhydon
Chansey
Tangela
Kangaskhan
Horsea
Seadra
Goldeen
Seakin

In [ ]:
# merge pokemon with non-essential different forms (many pikachus, mimikyu-busted etc.)

merge_list = [
            'Pikachu', 'Basculin', 'Aegislash', 'Pumpkaboo', 'Gourgeist', 'Zygarde', 'Minior', 'Mimikyu', 'Eiscue', 'Indeedee', 'Morpeko', 'Oinkologne',
            'Maushold', 'Dudunsparce', 'Squawkabily', 'Palafin', 'Tatsugiri', 'Cramorant', 'Zarude', 'Koraidon', 'Miraidon', 'Terapagos',
            'totem' # totem pokemon were included as different forms in the API
            ]

statement_merge = """MATCH (p:Pokemon)
WHERE p.name CONTAINS $name AND p.name <> $name
WITH collect(p) AS variants
MERGE (p2:Pokemon {name:$name})
SET p2.description = head([v IN variants WHERE v.description IS NOT NULL | v.description]),
    p2.number = head([v IN variants WHERE v.number IS NOT NULL | v.number])
FOREACH (v IN variants | DETACH DELETE v)
"""

with driver.session() as session:
    for pokemon in merge_list:
        session.execute_write(
                write_data,
                params_dict = {
                    'name': pokemon
                },
                statement = statement_merge
        )

In [ ]:
# fix issue of non-damaging moves having 'NaN' as power

statement_fix_move_power = """MATCH (m:Move)
WHERE toString(m.power) = 'NaN'
SET m.power=0
"""
with driver.session() as session:
    session.execute_write(
                write_data,
                params_dict = {},
                statement = statement_fix_move_power
    )